# Long Horizons: How the Error Grows with $T$

**Part 6 of a supervised research project — Dr Mathieu Gerber, University of Bristol**

> ### ⚠️ This notebook has not been run yet
>
> The code is complete and the sections explain what each experiment tests, but the
> **Conclusion** cells are placeholders. A full run takes roughly **35 minutes** (see the cost
> table in Section 0). Set `QUICK = True` for a two-minute smoke test first.

## Why this notebook exists

Dr Gerber's comment on Part 3:

> *"The time horizon $t = 2000$ that you consider is still too small; $t$ should be at least of
> order $10^6$ to have some confidence that the error does not increase over time when $M < d$."*

Running longer is the easy half. The harder half is that **the test used in Parts 1, 3 and 3b
cannot detect what he is describing, and gets worse as $T$ grows.**

Those notebooks split one run in half and compare the mean error in the two halves. Theory (Del
Moral & Gerber 2026, §7.5.1) says the error over a horizon cannot be of order smaller than
$\sqrt{\log T / N}$, and §5.5.1 states the $\sqrt{\log T}$ factor "cannot be removed". If the growth
really is $\sqrt{\log T}$, then the second-half / first-half ratio is

$$\frac{\sqrt{\log T}}{\sqrt{\log (T/2)}} \;\approx\; 1.05 \text{ at } T = 2000, \qquad
\approx 1.03 \text{ at } T = 10^{6}.$$

**It shrinks.** Because $\log$ saturates, a longer run makes the effect *harder* to see, not easier —
and Part 3's measured noise was already about $\pm 3\%$.

## What this notebook does differently

| | Parts 1, 3, 3b | here |
|---|---|---|
| statistic | **mean** error | **maximum** error over the horizon |
| comparison | two halves of **one** run | **whole runs** at $T = 10^2 \dots 10^6$ |
| predicted signal | $\times 1.05$, falling with $T$ | $\times 1.73$ from $10^2$ to $10^6$ |

The switch to the maximum matters as much as the switch to decades: §7.5.1's bound is on
$\max_{1 \leq t \leq T}$, and averaging smooths away exactly the spikes that are supposed to grow.

**A single-seed preview** at $d = 4$, $M = 2$, $\rho(F) = 0.9$ suggested the distinction is real —
`max_gap` went $0.879 \to 1.396 \to 1.739$ as $T$ went $10^4 \to 10^5 \to 10^6$, while `mean_gap`
stayed at $0.087$ throughout. That is one seed and one configuration, and the point of this notebook
is to measure it properly.

## Roadmap

| Section | Content |
|---|---|
| 0 | Setup, the cost table, and the `QUICK` switch |
| 1 | Why the half-run test cannot work — the arithmetic, made explicit |
| 2 | **The main sweep**: max and mean error across decades of $T$, three regimes, two spectral radii |
| 3 | Fitting against $\sqrt{\log T}$: flat, at the proven rate, or faster? |
| 4 | Does $N$ have to grow like $\log T$? (§7.5.2's prediction) |
| 5 | Summary |

Parts 1–5 are unchanged by anything here.

---
## Section 0 — Setup

`QUICK = True` runs a reduced grid in about two minutes, for checking the code end to end.
`QUICK = False` is the real run.

**Cost of the full run**, measured on this machine at $d = 4$, $M = 2$, $N = 1000$:

| $T$ | seconds per run | peak memory |
|---|---|---|
| $10^4$ | $1.2$ | $36$ MB |
| $10^5$ | $11.8$ | $68$ MB |
| $10^6$ | $116$ | $396$ MB |

Memory is comfortable, so no special machinery is needed — `run_case` is used unchanged. Time is the
constraint, which is why the number of seeds falls as $T$ rises.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

from filters import run_case

QUICK = True          # <-- set False for the real run (~35 min)

if QUICK:
    T_GRID  = [100, 1_000, 10_000]
    SEEDS   = {100: 4, 1_000: 4, 10_000: 2}
    N_GRID  = [250, 1000]
    T_N     = [1_000, 10_000]
else:
    T_GRID  = [100, 1_000, 10_000, 100_000, 1_000_000]
    SEEDS   = {100: 8, 1_000: 8, 10_000: 8, 100_000: 4, 1_000_000: 2}
    N_GRID  = [250, 1000, 4000]
    T_N     = [1_000, 10_000, 100_000]

CASES   = [(2, 2, "M = d"), (2, 4, "M > d"), (4, 2, "M < d")]
B_VALS  = [0.9, 1.0]
N_PART  = 1000
COUPLING = 0.3          # as in Parts 2-4
DATA_SEED = 42

CASE_COLOR = {"M = d": "tab:blue", "M > d": "tab:green", "M < d": "tab:red"}
B_STYLE    = {0.9: "-", 1.0: "--"}
B_MARKER   = {0.9: "o", 1.0: "s"}

plt.rcParams["figure.figsize"] = (10, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
warnings.filterwarnings("ignore", message=".*encountered in matmul")

print(f"QUICK = {QUICK}")
print(f"T grid   : {T_GRID}")
print(f"seeds    : {SEEDS}")
print(f"regimes  : {[c[2] for c in CASES]}")
print(f"rho(F)   : {B_VALS}   (coupling = {COUPLING}, N = {N_PART})")
est = sum(SEEDS[T] * len(CASES) * len(B_VALS) * (T / 1e6 * 116) for T in T_GRID)
print(f"\nrough estimate for Section 2: {est/60:.0f} min")

---
## Section 1 — Why the half-run test cannot work

Before running anything, it is worth making the arithmetic explicit, because it is the reason this
notebook exists rather than a longer version of Part 3.

Suppose the error over a horizon really does grow like $\sqrt{\log T}$, the rate §7.5.1 says cannot
be improved on. Two ways to look for it:

**(a) Split one run in half.** Compare the mean error over $[1, T/2]$ against $[T/2, T]$. The
expected ratio is $\sqrt{\log T} / \sqrt{\log(T/2)}$.

**(b) Compare whole runs across decades.** Run to $T = 10^2$ and to $T = 10^6$ and compare the
maxima. The expected ratio is $\sqrt{\log 10^6} / \sqrt{\log 10^2}$.

The cell below computes both. The second column is what Parts 1, 3 and 3b measured.

In [ ]:
print("(a) half-run ratio, if the growth is sqrt(log T):\n")
print(f"{'T':>12}{'2nd half / 1st half':>24}")
for T in (100, 2_000, 10_000, 10**6, 10**9):
    r = np.sqrt(np.log(T)) / np.sqrt(np.log(T / 2))
    print(f"{T:>12,}{r:>24.4f}")
print("\n  -> the signal SHRINKS as T grows, because log saturates.")
print("     Part 3's measured run-to-run noise was already about +/- 3%.")

print("\n(b) whole-run ratio across decades, same assumption:\n")
base = 100
print(f"{'T':>12}{'max error vs T=100':>24}")
for T in (100, 1_000, 10_000, 100_000, 10**6):
    r = np.sqrt(np.log(T)) / np.sqrt(np.log(base))
    print(f"{T:>12,}{r:>24.4f}")
print(f"\n  -> {np.sqrt(np.log(10**6))/np.sqrt(np.log(100)):.2f}x from 1e2 to 1e6: large enough to measure.")

print("\nAnd the second reason to change the statistic: the bound in Section 7.5.1 is on")
print("max_t, not on the mean. Averaging smooths exactly the spikes that are supposed to grow.")

**Conclusion.** *Placeholder — fill in after running.*

---
## Section 2 — The main sweep

Run each of the three regimes at both spectral radii, over the full range of horizons, and record
three things at each setting:

- **`max_gap`** — the largest PF–KF gap anywhere in the run. This is the quantity the theory bounds.
- **`mean_gap`** — the time-average, i.e. what Parts 1–3b reported. Kept for comparison.
- **`min_ess`** — does the particle cloud survive a million steps?

Averaged over the seeds listed in Section 0 (fewer at the long horizons, where each run costs two
minutes).

**What to look for.** If `max_gap` grows while `mean_gap` stays flat, the two statistics are
measuring different things and the choice of statistic explains why Parts 1–3b saw nothing. If both
stay flat, the error really is bounded in the strong sense over $10^6$ steps. The regime to watch is
$M < d$, which sits outside the assumptions of Del Moral & Gerber (2026) — condition (P1)b requires
$H$ bi-Lipschitz and hence $M \geq d$ — so it is the one case where the theory offers no guarantee.

In [ ]:
import time

results = {}          # (label, b) -> dict of arrays indexed like T_GRID
t_start = time.time()

for d, M, label in CASES:
    for b in B_VALS:
        mx, mn, ess = [], [], []
        for T in T_GRID:
            runs = [run_case(d, M, N=N_PART, T=T, b=b, coupling=COUPLING,
                             data_seed=DATA_SEED, pf_seed=s)
                    for s in range(SEEDS[T])]
            mx.append(np.mean([r["max_gap"] for r in runs]))
            mn.append(np.mean([r["mean_gap"] for r in runs]))
            ess.append(np.mean([r["min_ess"] for r in runs]))
            del runs
        results[(label, b)] = {"max": np.array(mx), "mean": np.array(mn),
                               "ess": np.array(ess)}
        print(f"  done {label:7s} rho={b}   ({time.time()-t_start:.0f}s elapsed)")

print(f"\ntotal {time.time()-t_start:.0f}s\n")

hdr = (f"{'regime':8s}{'rho':>5}" + "".join(f"{'T=' + f'{T:.0e}':>12}" for T in T_GRID))
for key, stat in (("max", "MAXIMUM gap"), ("mean", "MEAN gap"), ("ess", "min ESS")):
    print(f"\n{stat}")
    print(hdr); print("-" * len(hdr))
    for d, M, label in CASES:
        for b in B_VALS:
            v = results[(label, b)][key]
            print(f"{label:8s}{b:>5}" + "".join(f"{q:>12.4f}" for q in v))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
Ts = np.array(T_GRID, dtype=float)

for key, ax, ttl in (("max", axes[0], "maximum gap over the horizon"),
                     ("mean", axes[1], "mean gap (what Parts 1-3b reported)"),
                     ("ess", axes[2], "minimum ESS")):
    for d, M, label in CASES:
        for b in B_VALS:
            v = results[(label, b)][key]
            ax.semilogx(Ts, v, B_MARKER[b] + B_STYLE[b], color=CASE_COLOR[label],
                        lw=1.8, ms=5, alpha=0.9,
                        label=f"{label}, $\\rho={b}$")
    ax.set_xlabel("$T$ (log scale)"); ax.set_title(ttl, fontsize=10)
    ax.set_ylim(bottom=0)
axes[0].set_ylabel("gap"); axes[2].set_ylabel("ESS")
axes[0].legend(fontsize=7, ncol=2)
fig.suptitle("Section 2: the maximum and the mean behave differently as $T$ grows", y=1.0)
plt.tight_layout(); plt.show()

print("Growth from the shortest to the longest horizon:\n")
print(f"{'regime':8s}{'rho':>5}{'max gap':>22}{'mean gap':>22}")
for d, M, label in CASES:
    for b in B_VALS:
        r = results[(label, b)]
        print(f"{label:8s}{b:>5}"
              f"{r['max'][0]:>9.3f} -> {r['max'][-1]:>6.3f} ({r['max'][-1]/r['max'][0]:>4.2f}x)"
              f"{r['mean'][0]:>9.3f} -> {r['mean'][-1]:>6.3f} ({r['mean'][-1]/r['mean'][0]:>4.2f}x)")
print(f"\nFor reference, sqrt(log T) predicts "
      f"{np.sqrt(np.log(T_GRID[-1])/np.log(T_GRID[0])):.2f}x over this range.")

**Conclusion.** *Placeholder — fill in after running.*

---
## Section 3 — Fitting against $\sqrt{\log T}$

Section 2 shows whether the maximum grows. This section asks **how fast**, by plotting `max_gap`
against $\sqrt{\log T}$ and fitting a straight line through the origin.

Three outcomes, and they mean quite different things:

| fitted behaviour | interpretation |
|---|---|
| flat | the error is bounded in the strong sense; better than the theory requires |
| straight line in $\sqrt{\log T}$ | growth at exactly the rate §7.5.1 says is optimal — the bound is tight |
| **faster than $\sqrt{\log T}$** | **growth beyond the proven rate** — the interesting outcome, especially at $M < d$, where no guarantee applies |

The single-seed preview mentioned in the introduction suggested the third, but on one seed and one
configuration. A log–log fit of `max_gap` against $\log T$ gives the exponent directly: $0.5$ is the
predicted rate.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.4))
x = np.sqrt(np.log(np.array(T_GRID, dtype=float)))

print(f"{'regime':8s}{'rho':>5}{'exponent of log T':>20}{'(0.5 = predicted rate)':>26}")
print("-" * 59)
for d, M, label in CASES:
    for b in B_VALS:
        v = results[(label, b)]["max"]
        a1.plot(x, v, B_MARKER[b] + B_STYLE[b], color=CASE_COLOR[label], lw=1.8, ms=5,
                label=f"{label}, $\\rho={b}$")
        # exponent alpha in  max_gap ~ (log T)^alpha
        alpha = np.polyfit(np.log(np.log(np.array(T_GRID, dtype=float))), np.log(v), 1)[0]
        a2.plot(x, v / v[0], B_MARKER[b] + B_STYLE[b], color=CASE_COLOR[label], lw=1.8, ms=5,
                label=f"{label}, $\\rho={b}$  ($\\alpha={alpha:.2f}$)")
        print(f"{label:8s}{b:>5}{alpha:>20.3f}")

a2.plot(x, x / x[0], "k--", lw=1.5, label="predicted: $\\sqrt{\\log T}$")
a1.set_xlabel("$\\sqrt{\\log T}$"); a1.set_ylabel("maximum gap")
a1.set_title("Maximum gap against $\\sqrt{\\log T}$", fontsize=10); a1.legend(fontsize=7, ncol=2)
a2.set_xlabel("$\\sqrt{\\log T}$"); a2.set_ylabel("growth relative to the shortest run")
a2.set_title("The same, normalised, against the predicted rate", fontsize=10)
a2.legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.show()

print("\nalpha is the fitted exponent in  max_gap ~ (log T)^alpha.")
print("alpha ~ 0.5 matches the rate of Section 7.5.1; alpha > 0.5 is faster than the proven rate;")
print("alpha ~ 0 means no growth at all.")

**Conclusion.** *Placeholder — fill in after running.*

---
## Section 4 — Does $N$ have to grow like $\log T$?

A second, sharper prediction from the same paper, §7.5.2: for a **fixed** target precision, the
number of particles must grow logarithmically in the horizon,

$$N \;\geq\; C\,\big(1 + \log(1 + \rho^{-1})\big)^2 \rho^{-2}\,\big(1 + \log(T/q)\big),$$

and the paper states this dependence is **sharp** — it cannot be of order smaller than $\log(T/q)$.

That is a quantitative claim about the algorithm, not just a qualitative one, and it is cheaper to
test than Section 2. If the bound is tight, then $\text{max\_gap} \times \sqrt{N}$ should depend on
$T$ only through $\sqrt{\log T}$ — so curves for different $N$ should **collapse onto one line**
when plotted that way.

Run on the $M < d$ regime, the case the theory does not cover, at both spectral radii.

In [ ]:
d4, M4 = 4, 2
coll = {}
for b in B_VALS:
    for N in N_GRID:
        vals = []
        for T in T_N:
            runs = [run_case(d4, M4, N=N, T=T, b=b, coupling=COUPLING,
                             data_seed=DATA_SEED, pf_seed=s)
                    for s in range(min(4, SEEDS[T]))]
            vals.append(np.mean([r["max_gap"] for r in runs]))
            del runs
        coll[(b, N)] = np.array(vals)
    print(f"  done rho={b}")

xn = np.sqrt(np.log(np.array(T_N, dtype=float)))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.4))
for b in B_VALS:
    for N, c in zip(N_GRID, ("tab:blue", "tab:orange", "tab:red")):
        v = coll[(b, N)]
        a1.plot(xn, v, B_MARKER[b] + B_STYLE[b], color=c, lw=1.8, ms=5,
                label=f"$N={N}$, $\\rho={b}$")
        a2.plot(xn, v * np.sqrt(N), B_MARKER[b] + B_STYLE[b], color=c, lw=1.8, ms=5,
                label=f"$N={N}$, $\\rho={b}$")
a1.set_title("maximum gap", fontsize=10); a1.set_ylabel("max gap")
a2.set_title("the same $\\times \\sqrt{N}$ — should collapse", fontsize=10)
a2.set_ylabel("max gap $\\times \\sqrt{N}$")
for ax in (a1, a2):
    ax.set_xlabel("$\\sqrt{\\log T}$"); ax.legend(fontsize=7, ncol=2); ax.set_ylim(bottom=0)
fig.suptitle("Section 4: testing $\\max$ gap $\\sim \\sqrt{\\log T / N}$  ($M < d$)", y=1.0)
plt.tight_layout(); plt.show()

print(f"{'rho':>5}{'N':>7}" + "".join(f"{'T=' + f'{T:.0e}':>12}" for T in T_N)
      + f"{'  x sqrt(N) at largest T':>26}")
for b in B_VALS:
    for N in N_GRID:
        v = coll[(b, N)]
        print(f"{b:>5}{N:>7}" + "".join(f"{q:>12.4f}" for q in v)
              + f"{v[-1]*np.sqrt(N):>26.3f}")
print("\nIf the last column is roughly constant down the table, max gap ~ sqrt(log T / N) holds.")

**Conclusion.** *Placeholder — fill in after running.*

---
## Section 5 — Summary

*Placeholder — fill in after running.*

### Things to address when writing this up

- Whether `max_gap` grows while `mean_gap` stays flat, and therefore whether the statistic, rather
  than the horizon, is what Parts 1–3b were limited by.
- The fitted exponent $\alpha$ in each regime, against the predicted $0.5$. Particular attention to
  $M < d$, which lies outside condition (P1)b of Del Moral & Gerber (2026) and so carries no
  guarantee.
- Whether $\rho(F) = 1$ changes the exponent, or only the constant. Part 3b found the drift signs
  turned uniformly non-negative at the threshold, on three regimes and $T = 2000$ — this is the
  experiment that can say whether that meant anything.
- Whether $\max \text{gap} \times \sqrt{N}$ collapses, i.e. whether §7.5.2's $\log T$ dependence of
  $N$ holds in practice.
- Honest limits: one dataset per setting, and only two seeds at $T = 10^6$.

### Reference

Del Moral, P. & Gerber, M. (2026). *A perturbation theory for Feynman–Kac models and their particle
interpretations.* arXiv:2610.21587. — §5.5.1 on the $\sqrt{\log T}$ factor, §7.5.1 on the optimal
$(T, N)$ dependence, §7.5.2 on $N \propto \log T$.